# ⚡ Samtidiga Agentarbetsflöden med Microsoft Foundry (Python)

## 📋 Avancerad guide för parallell bearbetning

Denna anteckningsbok visar **samtidiga arbetsflödesmönster** med Microsoft Agent Framework. Du kommer att lära dig hur man bygger högpresterande, parallella arbetsflöden där flera AI-agenter körs samtidigt, vilket dramatiskt förbättrar genomflödet och möjliggör sofistikerade flertrådade affärsprocesser.

> **Migreringsnotis:** Detta exempel refererade tidigare till GitHub Models. GitHub Models är föråldrat (upphör juli 2026), så nu används **Microsoft Foundry** via `FoundryChatClient`, som riktar sig mot Azure OpenAI **Responses API**.

## 🎯 Lärandemål

### 🚀 **Grundläggande principer för samtidig bearbetning**
- **Parallell agentkörning**: Kör flera agenter samtidigt för maximal effektivitet
- **Orkestrering av arbetsflöden**: Koordinera samtidiga operationer samtidigt som datakonsistens upprätthålls
- **Prestandaoptimering**: Uppnå betydande hastighetsökning genom parallell bearbetning
- **Resurshantering**: Använd AI-modellresurser effektivt över samtidiga operationer

### 🏗️ **Avancerade samtidighetsmönster**
- **Fork-Join-bearbetning**: Dela upp arbete över flera agenter och sammanfoga resultat
- **Rörledningsparallellism**: Överlappande exekveringssteg för kontinuerligt genomflöde
- **Lastbalansering**: Jämnt fördela arbete över tillgängliga agentresurser
- **Synkroniseringspunkter**: Koordinera samtidiga agenter vid kritiska arbetsflödessteg

### 🏢 **Företagsapplikationer med samtidighet**
- **Högvolymdokumentbearbetning**: Bearbeta flera dokument samtidigt
- **Realtidsinnehållsanalys**: Samtidig analys av inkommande datastreams
- **Batchbearbetningsoptimering**: Maximera genomflöde för storskaliga operationer
- **Multimodal analys**: Parallell bearbetning av olika innehållstyper (text, bilder, data)

## ⚙️ Förutsättningar & installation

### 📦 **Nödvändiga beroenden**

Installera Agent Framework med stöd för samtidiga arbetsflöden:

```bash
pip install agent-framework -U
```

### 🔑 **Microsoft Foundry-konfiguration**

Logga in med Azure CLI (`az login`) så att `AzureCliCredential` kan autentisera, och ange sedan detaljerna för ditt Microsoft Foundry-projekt.

**Miljöinställningar (.env-fil):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

**Överväganden för samtidiga processer:**
- **Hastighetsgränser**: Övervaka Azure OpenAI:s hastighetsgränser för samtidiga förfrågningar
- **Resursanvändning**: Tänk på minnes- och CPU-användning med flera samtidiga agenter
- **Felh hantering**: Implementera robust felåterställning för parallella operationer

### 🏗️ **Arkitektur för samtidiga arbetsflöden**

```mermaid
graph TD
    A[Arbetsflöde Start] --> B[Samtidig Körning]
    B --> C[Agentpool 1]
    B --> D[Agentpool 2]
    B --> E[Agentpool 3]
    C --> F[Resultatsammanställning]
    D --> F
    E --> F
    F --> G[Slutligt Resultat]
    
    H[Microsoft Foundry] --> C
    H --> D
    H --> E
```

**Viktiga fördelar:**
- **⚡ Prestanda**: Betydande hastighetsökning genom parallell exekvering
- **📈 Skalbarhet**: Hantera ökande arbetsbelastningar utan proportionell tidsökning
- **🔄 Effektivitet**: Bättre nyttjande av tillgängliga datorkraftresurser
- **🎯 Genomflöde**: Bearbeta mer arbete på samma tid

## 🎨 **Designmönster för samtidiga arbetsflöden**

### 🔍 **Forsknings- och analysrörledning**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Arbetsflöde för databehandling**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Innehållsskapanderörledning**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Flerstegsprocess**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Företagsfördelar med prestanda**

### ⚡ **Genomflödesoptimering**
- **Parallell exekvering**: Flera agenter arbetar samtidigt
- **Resursutnyttjande**: Maximal effektivitet för tillgänglig AI-modellkapacitet
- **Tidsminskning**: Betydande minskning av total bearbetningstid
- **Skalbar arkitektur**: Lätt att lägga till fler samtidiga agenter vid behov

### 🛡️ **Tillförlitlighet och motståndskraft**
- **Fel tolerans**: Fel hos enskilda agenter stoppar inte hela arbetsflödet
- **Fel isolering**: Problem i en samtidigt gren påverkar inte andra
- **Graceful Degradation**: Systemet fortsätter fungera även med minskad agentkapacitet
- **Återställningsmekanismer**: Automatisk omförsök och felhantering för misslyckade operationer

### 📊 **Övervakning & observabilitet**
- **Spårning av samtidig exekvering**: Övervaka framsteg för alla parallella operationer
- **Prestandamått**: Mät hastighetsökning och effektivitetsvinster
- **Resursanvändningsanalys**: Optimera tilldelning av samtidiga agenter
- **Flaskhalsidentifiering**: Hitta och åtgärda prestandabegränsningar

Låt oss bygga högpresterande samtidiga AI-arbetsflöden! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = provider.as_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = provider.as_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)


In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
